There's already a website with every cards information, which we can retrieve for future use.

Currently, when doing a Performance Comparison, vanilla triggers or sentinels will create lots of outliers during analysis. 
It would be much easier if we could just check if a card is a vanilla trigger, or in a future stage of analysis, look for things of note in card text.

Given that we will want to look up this information often, it would be wise to store the data in a database so we don't need to do thousands of web lookups per analysis. Instead, we can do thousands of web lookups once, and maybe a few more every now and again, when the data doesn't appear in the database.



Since different cards can share a name, and card's can have effects that make them share names, or have extra names; we should include that information in a way for easy lookups, and avoid using card name's as ID's

We're already using a NOSQL database, so having a dictionary of card names for each card seems appropriate

idk how the website will be strucuted, so we should investigate skipping this step, but I know decklog has a link to each cards page, so as a last resort, we can take a decklog as input with out lookup function (form whicher analysis finds we're missing a cards info, while looking over a particular deck, we can just pass the decklog), and then find the cards section of the page, and follow the link. This seems awkward, but it is potentially reliable.

In [2]:

import card_data_collection_pipeline

import db_operations
import helpers


import requests

from bs4 import BeautifulSoup as Soup

In [18]:
URLS = [
    # Nubatama PG (clan info caused a probelm)
    # 'https://en.cf-vanguard.com/cardlist/?cardno=D-SS09/008EN-R',

    # 'https://en.cf-vanguard.com/cardlist/?cardno=D-PR/002EN',
    #Card with flavor, but no rules text (DE Crit \|/   PG /|\)
    # 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-SS08/032EN',

    #Sighned banGdream Crit
    # 'https://en.cf-vanguard.com/cardlist/?cardno=D-BT11/EX01EN-S',

    # Twincast
    'https://en.cf-vanguard.com/cardlist/?cardno=DZ-SS10/020EN',
    
 #   # Dual Nation Unit (Maple)
    # 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-SS08/FFR16EN',##
    #
#     # Dual Nation Trigger Unit (Grandpa)
    # 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-SS05/034EN',##

# #    # Dual Nation Order (Camera-chan)
    # 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-SS05/067EN',##

# # #Benerator
    # 'https://en.cf-vanguard.com/cardlist/?cardno=BCS2425/VGS04',##

#     # BlessFavor (trigger unit)
# 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-BT07/024EN',

# #     # Yuika (normal unit)
# # # # URL = 
# 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-SS08/Re49EN',

# # Vyrgillia (g3)
# 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-SS08/007EN',

# # # PG
# 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-LBT02/FR42EN', ##

# #     # Stil Nacht (blitz)
# #  # # URL = 
# 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-LBT02/019EN',

# #     # Silent Snowfall
# # # # URL = 
# 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-BT06/025EN',

# #     # Mygo music
# # # # URL = 
# 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-BT01/EX41EN',##

# #     # Buddy Fight Trigger order
# # # # URL = 
# 'https://en.cf-vanguard.com/cardlist/?cardno=DZ-TB01/077EN'
]
data_list = []
for url in URLS:
    request = requests.get(url)
    soup=Soup(request.text, 'html.parser')
    data = soup.find(attrs={'class':'data'})
    data_list.append(data)

In [19]:
for i, data in enumerate(data_list):
    print(type(data))

<class 'bs4.element.Tag'>


In [20]:
def prepare_test(data):
    name = data.find(attrs={'class':'name'}).text.strip()
    effect = data.find(attrs={'class':'effect'}).text.strip()
    flavor = data.find(attrs={'class':'flavor'}).text.strip()
    stats_line, bottom_line = data.find_all(attrs={'class':'text-list'})
    bottom_line = bottom_line.text.strip().split('\n')
    split_stats = stats_line.text.strip().split('\n')

    if len(stats_line.find_all(attrs={'class':'nation'})) == 2:
        second_nation = split_stats.pop(2)
        split_stats[1] = split_stats[1] + ' / ' + second_nation

    elif stats_line.find(attrs={'class':'group'}):
        clan = split_stats.pop(3)
        split_stats[2] = clan + ' / ' + split_stats[2]

    return [name, split_stats, effect, flavor, bottom_line]

In [22]:
prepared_tests = []
for i, data in enumerate(data_list):
    print(i)
    prepared_tests.append(prepare_test(data))

prepared_tests

0


[['Protection: Twincast',
  ['Blitz Order',
   '-',
   '-',
   'Grade 2',
   'Power ',
   'Critical -',
   'Shield ',
   'Regalis piece'],
  "Regalis Piece(You may only have one Regalis Piece in your deck, and use it a total of one time in a fight.)  Choose one of your units being attacked, and it gets [Power]+10000 until end of that battle.  [CONT]:This card's [AUTO] ability can be resolved even if you have resolved a Regalis Piece ability this fight. [AUTO](Drop):When your unit is attacked, [COST][remove this card from drop], choose one of your units being attacked, and it gets [Power]+5000 until end of that battle for each grade of that unit.",
  "We'll protect everyone! With a double stance!",
  ['Standard', 'DZ-SS10/020EN', 'TD', 'illust/匈歌ハトリ']]]

In [13]:

def extract_data_from_html(test):
    data_entry = dict()

    data_entry['name'] = test[0]


    # ~~~~~~~~~~~~~~~~~~~~~~~~~Section 1~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    card_type = test[1][0]
    # Dual nation cards are longer than expected, so we'll fit them into our zone here
    # if (len(test[1]) == 8 and "Order" in card_type and "Regalis" not in test[1][7])\
    # or (len(test[1]) == 9 and "Normal Unit" == card_type)\
    # or (len(test[1]) == 10 and "Trigger Unit" == card_type):
    #     second_nation = test[1].pop(2)
    #     test[1][1] = test[1][1] + ' / ' + second_nation

    data_entry['type'] = card_type
    data_entry['nation'] = test[1][1]
    data_entry['race'] = test[1][2]
    
    # Crests, Units, and orders each have a different set of attributes which need TLC
    if "Crest" in card_type:
        data_entry['grade'] = None
    else:
        data_entry['grade'] = int(test[1][3].split(' ')[1])

    if "Unit" in card_type:
        data_entry['power'] = int(test[1][4].split(' ')[1])
        data_entry['crit'] = int(test[1][5].split(' ')[1])

        # Grade 3's have an empty string when checking for shield, so this fixes that bug
        shield = test[1][6].split(' ')[1]
        if shield == '' or '-' == shield:
            data_entry['shield'] = None
        else:
            data_entry['shield'] = int(shield)
        data_entry['shield'] = test[1][6]

        data_entry['ability'] = test[1][7]    

    else: # If "Order" in card_type
        data_entry['power'] =   None
        data_entry['crit'] =    None
        data_entry['shield'] =  None
        data_entry['ability'] = None

    if "Trigger" in card_type:
        data_entry['trigger'] = test[1][8]
    else:
        data_entry['trigger'] = None

    #~~~~~~~~~~~~~~~~~~~~Section 2~~~~~~~~~~~~~~~~~~~~~~~~~~
    data_entry['effect'] = test[2]

    #~~~~~~~~~~~~~~~~~~~~Section 3~~~~~~~~~~~~~~~~~~~~~~~~~
    data_entry['flavor'] = test[3]

    #~~~~~~~~~~~~~~~~~~~~Section 4~~~~~~~~~~~~~~~~~~~~~~~
    data_entry['format'] = test[4][0]
    data_entry['id'] = test[4][1]
    data_entry['rarity'] = test[4][2]
    data_entry['artist'] = test[4][3]

    return data_entry

In [14]:
print(prepared_tests)

[['Stealth Dragon, Utsuroi', ['Normal Unit', 'Dragon Empire', 'Nubatama / Abyss Dragon', 'Grade 1', 'Power 6000', 'Critical 1', 'Shield 0', 'Boost'], '[CONT]:Sentinel （You may only have up to four cards with "[CONT]:Sentinel" in a deck）[AUTO]:When this unit is put on (GC), choose one of your units, and it cannot be hit until end of that battle. If your hand has two or more cards, choose a card from hand, and discard it.', 'These blade-shattering chains shall chain down the will to counterattack.', ['Standard', 'D-SS09/008EN-R', 'TDR', 'design:HMK84 illust:茄子乃\u3000icon:MAMEX']], ['Twin Buckler Dragon', ['Normal Unit', 'Dragon Empire', 'Plasma Dragon', 'Grade 1', 'Power 6000', 'Critical 1', 'Shield 0', 'Boost', '-'], '[CONT]:Sentinel (You may only have up to four cards with "[CONT]:Sentinel" in a deck)[AUTO]:When this unit is put on (GC), choose one of your units, and it cannot be hit until end of that battle. If your hand has two or more cards, choose a card from your hand, and discard

In [15]:
data = None
for foo in prepared_tests:
    data = extract_data_from_html(foo)
    if data['format'] == "S": raise LookupError
    print(data)

{'name': 'Stealth Dragon, Utsuroi', 'type': 'Normal Unit', 'nation': 'Dragon Empire', 'race': 'Nubatama / Abyss Dragon', 'grade': 1, 'power': 6000, 'crit': 1, 'shield': 'Shield 0', 'ability': 'Boost', 'trigger': None, 'effect': '[CONT]:Sentinel （You may only have up to four cards with "[CONT]:Sentinel" in a deck）[AUTO]:When this unit is put on (GC), choose one of your units, and it cannot be hit until end of that battle. If your hand has two or more cards, choose a card from hand, and discard it.', 'flavor': 'These blade-shattering chains shall chain down the will to counterattack.', 'format': 'Standard', 'id': 'D-SS09/008EN-R', 'rarity': 'TDR', 'artist': 'design:HMK84 illust:茄子乃\u3000icon:MAMEX'}
{'name': 'Twin Buckler Dragon', 'type': 'Normal Unit', 'nation': 'Dragon Empire', 'race': 'Plasma Dragon', 'grade': 1, 'power': 6000, 'crit': 1, 'shield': 'Shield 0', 'ability': 'Boost', 'trigger': None, 'effect': '[CONT]:Sentinel (You may only have up to four cards with "[CONT]:Sentinel" in 

Alright, looks like we got the information retrival ironed out... for now :/
However, when working with saving the data, I noticed sometimes there were issues. The re-write form keeps getting added over and over, so it's probably not getting read? lets investiagae

In [16]:
TEST = 'DZ-SS08/007EN : Vyrgilla, "Rewrite Form: Alfknights"'
def db_lookup(id_and_name):
    id = helpers.extract_card_id(id_and_name)
    return db_operations.find_first_in_table('main_table', 'card_data', {'id':id})


def url_lookup(id_and_name):
    id = helpers.extract_card_id(id_and_name)
    return card_data_collection_pipeline.add_card_info_to_db(id)


In [17]:
url_lookup(TEST)
test = db_lookup(TEST)

print( test )

{'_id': ObjectId('69b2a7866578a8e01b164ab3'), 'name': 'Vyrgilla, "Rewrite Form: Alfknights"', 'type': 'Normal Unit', 'nation': 'Dragon Empire', 'race': 'Cyber Dragon', 'grade': 3, 'power': 13000, 'crit': 1, 'shield': None, 'ability': 'Twin Drive!!, Persona Ride', 'trigger': None, 'effect': '[Rewrite] - [ACT](Hand):If your opponent\'s vanguard is grade 3 or greater, [COST][bind a "Rewrite the Star, Vyrgilla" on (VC)], ride this card as [Stand], and perform all of the following. At the end of that turn, ride the card bound for this cost as [Rest]. ・Search your deck or drop for up to one <Galaxian>, call it to (RC), search your deck or drop for up to one Ultimate Skill card, reveal it and put it into hand, and if you searched the deck, shuffle the deck. ・Until end of turn, this unit gets "[AUTO](VC)[1/Turn]:When this unit attacks, [COST][Counter-Blast 1], [Stand] all of your front row rear-guards, and this unit gets drive +1 until end of that battle.", and all of your front row units get 

In [ ]:



# def add_card_info_to_db(url: str) -> bool:
#     """Given a card name, add its information to a central data base"""

#     try:
#         html_text = get_card_info_from_bushi_site(url)

#         data_entry = exctract_info_from_bushi_site(html_text)

#         db_operations.insert_one_into_table('main_table',
#                                             'card_data',
#                                             data_entry)

#         return True

#     except Exception as something_went_wrong:
#         print(something_went_wrong)
#         return False


# def get_card_info_from_bushi_site(url: str) -> list[list[str]]:
#     """Given a name, and maybe decklog, retrieve the text of the official webpage with it's information
    
#     The wepbage contains an element, 'data', which has all the informatin about the card for the URL.
#     It contains some sub elements, so the easiest way to extract it is to simply retrieve the text from each,
#     and put it into a big list.

#     There are lots of different atributes of the card that are contained in one element, seperated by `\n`
#     new line charachters. So, for each element, we'll turn it into a list of attributes,
#     ending us with a list of lists of string. list [ list [ str ] ]

#     # name 
#     # stats
#     # effect 
#     # flavor
#     # bottomline(set, artist, etc.)

#     These are the 5 children of the data element, each with a different number of attributes.
#     """
#     response = requests.get(url)
#     soup=Soup(response.text, 'html.parser')
#     data_element = soup.find(attrs={'class':'data'})

#     data = []
#     for child in data_element:
#         text = child.text
#         if text.strip() != '':
#             data.append(text.strip())

#     return data


# def exctract_info_from_bushi_site(seperated_soup: list[list[str]]) -> dict:
#     """Given the text from a card's bushi site page, return the information we want in our database
    
#     We hit a few snags, due to the combinations of order, trigger, and unit atributes that can cause
#     some headache.

#     For now, Order type (Music, Codex, Fox Art, Etc.) are placed in the 'race' section,
#     and orders with no type just have '-' as their race. We can wrange these issues later.
#     At least we have the data.
#     """

#     data_entry = dict()
#     test = []
#     for i, item in enumerate(seperated_soup):
#         if i == 3: test.append(item) # We want to preserve line breaks in flavor text?
#         else: test.append(item.split('\n'))

#     card_type = test[1][0]

#     data_entry['name'] = test[0][0]

#     data_entry['type'] = card_type
#     data_entry['nation'] = test[1][1]
#     data_entry['race'] = test[1][2]
#     data_entry['grade'] = int(test[1][3].split(' ')[1])
#     # ~~~~~~~~~~~~~~~~~~   UNIT / ORDER / TRIGGER LOGIC ~~~~~~~~~~~~~~~~~~~~~~~
#     if "Unit" in card_type:
#         data_entry['power'] = int(test[1][4].split(' ')[1])
#         data_entry['crit'] = int(test[1][5].split(' ')[1])
#         # Grade 3's have an empty string when checking for shield, so this fixes that bug
#         shield = test[1][6].split(' ')[1]
#         if shield == '':
#             data_entry['shield'] = 0
#         else:
#             data_entry['shield'] = int(shield)

#         data_entry['ability'] = test[1][7]

#     else:
#         data_entry['power'] = "None"
#         data_entry['crit'] = "None"
#         data_entry['shield'] = "None"
#         data_entry['ability'] = "None"

#     if "Trigger" in card_type:
#         data_entry['trigger'] = test[1][8]
#     else:
#         data_entry['trigger'] = "None"

#     data_entry['effect'] = test[2]
#     # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

#     # Some cards don't have flavor text, and this 'Magic Number Hack' inserts an empty string
#     if len(test) == 4:
#         test.insert(3, '')
#     data_entry['flavor'] = test[3]

#     data_entry['format'] = test[4][0]
#     data_entry['id'] = test[4][1]
#     data_entry['rarity'] = test[4][2]
#     data_entry['illust'] = test[4][3]

#     return data_entry
